# Solar-activity dependence of continuum airglow with nsb-py

The PALACE continuum model uses the F10.7 solar radio flux as an input describing
solar activity. This notebook varies that input while keeping the observer,
sky direction and UTC instant fixed. The calculation isolates the model's
solar-activity dependence at one Paranal geometry.

The example uses the public **nsb-py 0.2.0** API to:

1. Define a Paranal observer, an ICRS direction and a UTC instant.
2. Select **PALACE continuum only** and evaluate six explicit F10.7 inputs.
3. Compare the predicted **300–650 nm photon radiance** in a table and figure.

Only the Airglow continuum is included. Oxygen lines, zodiacal light, integrated
starlight and moonlight are excluded. The result describes a model response to
controlled inputs; it does not measure agreement with observed sky spectra.
For that assessment, see [observational validation](observational-validation.ipynb). For point queries
and observation-window planning, see [the CTAO API example](ctao-sky-background.ipynb).

Code: AGPL-3.0-only, Copyright (C) 2026 Vallés Puig, Ramon. Upstream data retain their source terms.


## 1. Prepare the Python environment

Select a fresh **CPython 3.12 or 3.13** Jupyter or Google Colab kernel and
**Run All**. The next cell installs **`nsb-py==0.2.0` from PyPI** and this
notebook's dependencies using the current kernel's Python. No NSB checkout,
GitHub installation of NSB or Rust toolchain is needed; a compatible NSB wheel
is required. The distribution is `nsb-py`, imported as `nsb`; the unrelated
package named `nsb` is rejected.

The package list is visible in `REQUIREMENTS`. Matching installed versions are
reused. If an already imported package changes, setup stops and asks you to
**restart the kernel and Run All**. Installation failures stop execution with
pip's error log; check network access and that you selected a supported kernel.

Results are generated inline using bundled NSB assets; no files are exported.


In [ ]:
# Run this cell first, using a CPython 3.12 or 3.13 kernel.
import importlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys

if sys.implementation.name != "cpython" or sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("Select a CPython 3.12 or 3.13 kernel, then Run All.")

# nsb-py is the distribution; nsb is its Python import. Do not install 'nsb'.
REQUIREMENTS = ['nsb-py==0.2.0', 'matplotlib>=3.8,<4']
try:
    metadata.distribution("nsb")
except metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("Remove the unrelated 'nsb' distribution with this kernel's pip, then restart.")

# Remember loaded dependencies so changes cannot silently leave stale imports.
loaded = {name: metadata.version(name)
          for module, names in metadata.packages_distributions().items()
          if module in sys.modules for name in names}
pip = [sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple"]
try:
    nsb_dist = metadata.distribution("nsb-py")
    direct = json.loads(nsb_dist.read_text("direct_url.json") or "{}")
except metadata.PackageNotFoundError:
    nsb_dist, direct = None, {}
if nsb_dist is not None and (nsb_dist.version != "0.2.0" or direct):
    if "nsb" in sys.modules:
        raise RuntimeError("Restart the kernel before selecting nsb-py==0.2.0 from PyPI.")
    subprocess.run(pip + ["--only-binary=:all:", "--no-deps", "--force-reinstall", "nsb-py==0.2.0"], check=True)
subprocess.run(pip + ["--only-binary=nsb-py", *REQUIREMENTS], check=True)

importlib.invalidate_caches()
changed = [name for name, version in loaded.items() if metadata.version(name) != version]
if changed:
    raise RuntimeError(f"Updated already-imported packages: {changed}. Restart the kernel, then Run All.")

# Check that the module resolves to the nsb-py installation, rather than a local file.
nsb_dist = metadata.distribution("nsb-py")
nsb_spec = importlib.util.find_spec("nsb")
owned = {Path(nsb_dist.locate_file(f)).resolve() for f in nsb_dist.files or []}
if nsb_dist.version != "0.2.0" or nsb_spec is None or not nsb_spec.origin or Path(nsb_spec.origin).resolve() not in owned:
    raise RuntimeError("Expected the nsb-py==0.2.0 module. Remove shadowing nsb.py/directories and restart.")
print("Ready: nsb-py", nsb_dist.version)


In [ ]:
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nsb
import siderust


## 2. Set the fixed geometry and solar inputs

The observer is at **Paranal**, with east-positive longitude **−70.4045°**, latitude
**−24.6272°** and height **2635 m**. The target is the fixed ICRS direction
**RA = 10°, Dec = −20°**, evaluated at **05:00 UTC on 15 September 2023**.

The six explicit F10.7 inputs span **67–166 sfu**. Observer, target, date, time,
model and wavelength band remain unchanged between evaluations. The
`AIRGLOW` component mask selects Airglow; choosing `PARANAL_PALACE_V1` in the
next cell restricts it to the continuum. In v0.2.0, Automatic would include
experimental Oxygen emission, so the continuum selection is explicit here.


In [ ]:
observer = siderust.Observer(lon_deg=-70.4045, lat_deg=-24.6272, height_m=2635.0)
instant = datetime(2023, 9, 15, 5, tzinfo=timezone.utc)
target = siderust.Direction(ra_deg=10.0, dec_deg=-20.0)
query = nsb.PointQuery(observer, instant, target, components=nsb.ComponentMask.AIRGLOW)
solar_flux = [67.0, 80.0, 100.0, 120.0, 140.0, 166.0]  # sfu
unit = "photons cm⁻² ns⁻¹ sr⁻¹"


## 3. Calculate the continuum radiance

For each solar input, construct the same continuum model with the corresponding
F10.7 value. `integrate_band` returns the photon radiance over 300–650 nm, in
**photons cm⁻² ns⁻¹ sr⁻¹**. The explicit solar input overrides the model's native
solar-history resolver for this calculation.


In [ ]:
radiance = []
for sfu in solar_flux:
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)
              .with_solar_radio_flux_sfu(sfu))
    evaluator = nsb.NsbEvaluator(config)
    radiance.append(evaluator.integrate_band(query, 300.0, 650.0))

table = "| F10.7 [sfu] | Continuum radiance |\n|---:|---:|\n"
table += "".join(f"| {sfu:.1f} | {value:.6f} |\n" for sfu, value in zip(solar_flux, radiance))
display(Markdown(table + f"\nRadiance units: {unit}; 300–650 nm."))


## 4. Compare the solar-input response

Each point is a separate model evaluation at the fixed geometry. The horizontal
axis shows the supplied F10.7 value; the vertical axis shows the band-integrated
continuum radiance. Connecting lines help compare the six evaluations and do not
represent a measured time series. The figure shows how the selected model responds
to solar input, without fitting any observations.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
ax.plot(solar_flux, radiance, "o-")
ax.set(xlabel="F10.7 solar radio flux [sfu]",
       ylabel=f"Continuum radiance [{unit}]",
       title="PALACE continuum, 300–650 nm, fixed Paranal geometry")
ax.grid(alpha=0.2)
plt.show()


## 5. Interpret and adapt the example

The continuum radiance increases across the supplied F10.7 values for this
configuration. Holding geometry and time fixed isolates this modeled dependence;
it does not establish the accuracy of the climatology on an individual night
or at another site. The calculation supplies no observational uncertainty estimate.

To examine another case, edit the observer, ICRS direction, UTC instant or solar
inputs, then rerun the notebook in a fresh kernel. Changing the component mask or
selecting Oxygen changes which physical emissions are included. Detector counts
and photometric magnitudes require their own instrument or passband treatment.

For model assumptions and domain limits, see the
[Airglow guide](../docs/nsb_components/airglow/README.md),
[restricted-band API contract](../docs/specifications/restricted-band-radiance.md)
and [scientific validation guide](../docs/scientific-validation.md).
